# collecting pitchfork review text

Get the full review text for every album in our merged Pitchfork + Discogs dataset (`pitchfork_discogs_initial.csv`).

current workflow:
- Load our albums and build matching keys
- Load the nolanbconaway Pitchfork dataset (reviews + text, 1999 to Jan 2017)
- Match our albums to those reviews
- Scrape the reviews that are still missing from pitchfork.com
- Combine, check, and save

## Loading album and building matching keys

We only need the columns that identify a review (artist, album, date, score, author, genre). We can always join back to the full table later on artist + album.

In [1]:
import pandas as pd

# the group's shared name cleaner (same one used for the Discogs matching)
from text_norm import normalize

In [2]:
albums = pd.read_csv("pitchfork_discogs_initial.csv")
print(albums.shape)

cols = ["artist", "album", "reviewdate", "score", "reviewauthor", "genre"]
albums = albums[cols].copy()
albums.head()

(10764, 32)


,artist,album,reviewdate,score,reviewauthor,genre
0,Studio 1,Studio 1,February 18 2009,8.5,Andy Battaglia,Electronic
1,John Fahey,The Great Santa Barbara Oil Slick,February 13 2005,8.2,Mark Richardson,Folk/Country
2,Reigning Sound,Too Much Guitar,August 19 2004,8.3,Stephen M. Deusner,Electronic
3,Mac Miller,Swimming,August 3 2018,7.5,Evan Rytlewski,Rap
4,The Essex Green,Essex Green EP,May 31 2000,8.1,Matt LeMay,Rock


`reviewdate` is text like "February 18 2009". Converting it lets us compare dates with the other dataset.

In [3]:
albums["reviewdate"] = pd.to_datetime(albums["reviewdate"], format="%B %d %Y", errors="coerce")

# errors="coerce" turns anything it can't read into NaT, so check how many failed
print("dates that didn't parse:", albums["reviewdate"].isna().sum())
print("earliest:", albums["reviewdate"].min())
print("latest:", albums["reviewdate"].max())

dates that didn't parse: 0
earliest: 1999-02-23 00:00:00
latest: 2019-12-31 00:00:00


`normalize()` lowercases, strips accents and punctuation, drops bracketed bits like "(Deluxe Edition)", a leading "the", and a trailing "EP"/"LP". so even if we have two datasets that spell a name slightly differently should end up with the same key.

In [4]:
albums["artist_key"] = albums["artist"].apply(normalize)
albums["album_key"] = albums["album"].apply(normalize)

# look at some rows where normalizing actually changed the name
changed = albums[albums["album"].str.lower() != albums["album_key"]]
print("albums whose key differs from the lowercase title:", len(changed))
changed[["artist", "artist_key", "album", "album_key"]].sample(10, random_state=1)

albums whose key differs from the lowercase title: 2545


,artist,artist_key,album,album_key
291,Faunts,faunts,Feel. Love. Thinking. Of.,feel love thinking of
445,Gaye Su Akyol,gaye su akyol,Hologram İmparatorluğu,hologram imparatorlugu
7796,Tamaryn,tamaryn,The Waves,waves
4666,William Basinski,william basinski,The Garden of Brokenness,garden of brokenness
4584,Sleaford Mods,sleaford mods,T.C.R. EP,t c r
3618,Virtual Self,virtual self,Virtual Self EP,virtual self
19,Teenage Fanclub,teenage fanclub,Man-Made,man made
1256,xxxy,xxxy,Everything EP,everything
7146,Modest Mouse,modest mouse,The Moon & Antarctica,moon and antarctica
2897,Goon Moon,goon moon,I Got a Brand New Egg Layin' Machine,i got a brand new egg layin machine


In [5]:
# empty keys would match nothing (or everything), so we want zero here
print("empty artist keys:", (albums["artist_key"] == "").sum())
print("empty album keys:", (albums["album_key"] == "").sum())

# albums that share a key -> we'll need the date to tell them apart later
dupes = albums[albums.duplicated(["artist_key", "album_key"], keep=False)]
print("rows sharing an artist+album key:", len(dupes))
dupes.sort_values(["artist_key", "album_key"]).head(10)

empty artist keys: 7
empty album keys: 0
rows sharing an artist+album key: 14


,artist,album,reviewdate,score,reviewauthor,genre,artist_key,album_key
1181,High Places,High Places EP,2007-08-03,8.2,Mike Powell,Electronic,high places,high places
2261,High Places,High Places,2008-09-24,8.0,Mark Richardson,Electronic,high places,high places
1948,Interpol,Interpol,2010-09-14,4.6,Nitsuh Abebe,Rock,interpol,interpol
4847,Interpol,Interpol EP,2002-07-16,9.3,Eric Carr,Rock,interpol,interpol
891,Lambchop,Is a Woman EP,2002-10-22,7.2,William Bowers,Rock,lambchop,is a woman
6666,Lambchop,Is a Woman,2002-03-03,5.8,Rob Mitchum,Rock,lambchop,is a woman
878,Maxwell,blackSUMMERS’night,2016-07-12,8.4,Rebecca Haithcoat,Pop/R&B,maxwell,blacksummersnight
3207,Maxwell,BLACKsummers'night,2009-07-10,7.8,David Drake,Pop/R&B,maxwell,blacksummersnight
5417,Serengeti,Kenny Dennis LP,2013-06-25,7.8,Nate Patrin,Rap,serengeti,kenny dennis
7284,Serengeti,Kenny Dennis EP,2012-03-27,7.8,Jeff Weiss,Rap,serengeti,kenny dennis


That dataset stops at January 8, 2017. i scraped other data later down

In [6]:
CUTOFF = pd.Timestamp("2017-01-08")

albums["before_cutoff"] = albums["reviewdate"] <= CUTOFF
print(albums["before_cutoff"].value_counts())

# reviews per year, to see where the post-cutoff albums fall
albums["reviewdate"].dt.year.value_counts().sort_index()

before_cutoff
True     9144
False    1620
Name: count, dtype: int64


reviewdate
1999     60
2000    125
2001    199
2002    447
2003    537
2004    552
2005    586
2006    585
2007    580
2008    587
2009    594
2010    638
2011    627
2012    632
2013    642
2014    581
2015    587
2016    581
2017    587
2018    537
2019    500
Name: count, dtype: int64

## loading the Pitchfork review text

we're using Nolan Conaway's **24,169 Pitchfork Reviews** dataset (Kaggle). it runs from Jan 1999 to Dec 2021, so it should cover all of our albums. i have it saved as data.sqlite3

How the database is organized:
- `reviews`: one row per review page (`review_url`, `pub_date`, `body` = the full text)
- `tombstones`: one row per *album* in a review (`title`, `score`, `best_new_music`). Most reviews cover one album, but some (box sets, double reviews) cover several.
- `artists` / `artist_review_map`, `author_review_map`: who the review is about, and who wrote it

In [7]:
import sqlite3

conn = sqlite3.connect("data.sqlite3")

# list the tables (and views) in the database
pd.read_sql("select name, type from sqlite_master where type in ('table', 'view')", conn)

,name,type
0,artists,table
1,reviews,table
2,artist_review_map,table
3,author_review_map,table
4,genre_review_map,table
5,tombstones,table
6,tombstone_label_map,table
7,tombstone_release_year_map,table
8,reviews_flat,view
9,standard_reviews_flat,view


The database has a ready-made `reviews_flat` view, but for multi-album reviews it glues titles together ("Album A, Album B") and some real titles have commas in them, so we can't split them back apart. Instead we start from `tombstones` (one row per album, with its own score) and attach the review text, date, artists, and author to each row.

In [8]:
query = """
select
    t.review_tombstone_id,
    t.review_url,
    t.title,
    t.score,
    t.best_new_music,
    r.pub_date,
    r.is_standard_review,
    a.artists,
    au.authors,
    r.body
from tombstones t
join reviews r on t.review_url = r.review_url
left join (
    select m.review_url, group_concat(ar.name, ', ') as artists
    from artist_review_map m
    join artists ar on m.artist_id = ar.artist_id
    group by m.review_url
) a on t.review_url = a.review_url
left join (
    select review_url, group_concat(author, ', ') as authors
    from author_review_map
    group by review_url
) au on t.review_url = au.review_url
"""

p4k = pd.read_sql(query, conn)
print(p4k.shape)
p4k.drop(columns="body").head()

(25212, 10)


,review_tombstone_id,review_url,title,score,best_new_music,pub_date,is_standard_review,artists,authors
0,/reviews/albums/9232-vesikansi/-0,/reviews/albums/9232-vesikansi/,Vesikansi,7.1,0.0,2006-07-27 06:00:04,1,Avarus,Matthew Murphy
1,/reviews/albums/22823-elsewhere/-0,/reviews/albums/22823-elsewhere/,Elsewhere,7.5,0.0,2017-01-27 06:00:00,1,Pinegrove,Sam Sodomsky
2,/reviews/albums/4703-le-tigre/-0,/reviews/albums/4703-le-tigre/,Le Tigre,8.5,NaN,1999-10-26 05:00:03,1,Le Tigre,Kristin Sage Rockermann
3,/reviews/albums/21978-stranger-to-stranger/-0,/reviews/albums/21978-stranger-to-stranger/,Stranger to Stranger,7.2,0.0,2016-06-09 05:00:00,1,Paul Simon,Mike Powell
4,/reviews/albums/11996-everyone-is-crying-out-t...,/reviews/albums/11996-everyone-is-crying-out-t...,"Everyone Is Crying Out to Me, Beware",7.4,0.0,2008-07-23 06:00:05,1,Alina Simone,Joshua Klein


`pub_date` looks like "2019-09-07 05:00:00". the 05:00 is just midnight Eastern time stored in UTC, so we only keep the date part. then i build `artist_key` / `album_key` with the same `normalize()` so both datasets are cleaned the same way.

In [9]:
p4k["pub_date"] = pd.to_datetime(p4k["pub_date"]).dt.normalize()

p4k["artist_key"] = p4k["artists"].apply(normalize)
p4k["album_key"] = p4k["title"].apply(normalize)

print("earliest:", p4k["pub_date"].min())
print("latest:", p4k["pub_date"].max())

earliest: 1999-01-05 00:00:00
latest: 2021-12-12 00:00:00


In [10]:
print("albums (tombstones):", len(p4k))
print("review pages:", p4k["review_url"].nunique())
print("review pages with more than one album:", (p4k["review_url"].value_counts() > 1).sum())

print("missing artists:", p4k["artists"].isna().sum())
print("empty artist keys:", (p4k["artist_key"] == "").sum())
print("empty album keys:", (p4k["album_key"] == "").sum())
print("empty review text:", (p4k["body"].str.strip() == "").sum())

albums (tombstones): 25212
review pages: 24169
review pages with more than one album: 586
missing artists: 0
empty artist keys: 10
empty album keys: 11
empty review text: 0


In [11]:
# how long are reviews? (in words)
p4k["word_count"] = p4k["body"].str.split().str.len()
p4k["word_count"].describe()

count    25212.000000
mean       702.540854
std        310.528845
min          1.000000
25%        521.000000
50%        631.000000
75%        784.000000
max       3591.000000
Name: word_count, dtype: float64

In [12]:
# does it cover the years our albums are from (1999-2019)?
p4k["pub_date"].dt.year.value_counts().sort_index()

pub_date
1999     220
2000     351
2001     469
2002     930
2003    1085
2004    1180
2005    1262
2006    1255
2007    1248
2008    1257
2009    1234
2010    1218
2011    1217
2012    1251
2013    1233
2014    1234
2015    1203
2016    1370
2017    1298
2018    1290
2019    1209
2020    1160
2021    1038
Name: count, dtype: int64

## matching our albums to the reviews

1. **exact**: same artist key + album key, review date within 3 days
2. **album+score**: same album key + same score, date within 3 days (artist written differently, e.g. "&" vs ",")
3. **fuzzy**: same score, date within 3 days, and "artist album" text at least 70% similar

the date window keeps same-named albums apart. the score is a strong extra check: two different reviews almost never share both the score and the date.

In [14]:
from difflib import SequenceMatcher

# give every one of our albums an id so we can track which ones are matched
albums["album_id"] = albums.index

# p4k without the big text column, and with its score renamed so it doesn't clash with ours
p4k_meta = p4k.drop(columns="body").rename(columns={"score": "score_p4k"})

DAYS = 3  # how far apart the two review dates can be


def keep_best(m, method):
    """Keep candidates within the date window, then one review per album and one album per review."""
    m = m.copy()
    m["days_off"] = (m["reviewdate"] - m["pub_date"]).dt.days.abs()
    m = m[m["days_off"] <= DAYS]
    m = m.sort_values("days_off")
    m = m.drop_duplicates("album_id").drop_duplicates("review_tombstone_id")
    m["match_method"] = method
    return m


def same_score(m):
    return (m["score"] - m["score_p4k"]).abs() < 0.05

### exact artist + album

In [15]:
m1 = albums.merge(p4k_meta, on=["artist_key", "album_key"])
m1 = keep_best(m1, "exact")

print("matched:", len(m1), "of", len(albums))
print("score disagrees:", (~same_score(m1)).sum())
print("days between the two dates:")
print(m1["days_off"].value_counts().sort_index())

matched: 10723 of 10764
score disagrees: 0
days between the two dates:
days_off
0    10722
2        1
Name: count, dtype: int64


In [16]:
# look at the exact matches where the score doesn't agree (if any)
m1.loc[~same_score(m1), ["artist", "album", "reviewdate", "score", "artists", "title", "pub_date", "score_p4k"]]

,artist,album,reviewdate,score,artists,title,pub_date,score_p4k


### same album + same score (artist written differently)

In [17]:
used_albums = set(m1["album_id"])
used_reviews = set(m1["review_tombstone_id"])

left = albums[~albums["album_id"].isin(used_albums)]
pool = p4k_meta[~p4k_meta["review_tombstone_id"].isin(used_reviews)]

m2 = left.merge(pool, on="album_key", suffixes=("", "_p4k"))
m2 = m2[same_score(m2)]
m2 = keep_best(m2, "album+score")

print("matched:", len(m2))
m2[["artist", "artists", "album", "title", "score", "days_off"]].head(20)

matched: 39


,artist,artists,album,title,score,days_off
0,Dean Blunt and Inga Copeland,"Inga Copeland, Dean Blunt",Black Is Beautiful,Black Is Beautiful,7.4,0
21,Ghostface Killah,"Ghostface Killah, Apollo Brown",The Brown Tape,The Brown Tape,6.0,0
22,Ryan Adams & the Cardinals,"Ryan Adams, The Cardinals",Jacksonville City Nights,Jacksonville City Nights,7.7,0
23,Green Velvet,"Green Velvet, Pajo",Whatever,Whatever,7.0,0
24,Loren Connors & David Grubbs,"Loren Connors, David Grubbs",Arborvitae,Arborvitae,6.4,0
25,KING,We Are KING,We Are KING,We Are KING,7.8,0
26,Loke Rahbek & Frederik Valentin,"Loke Rahbek, Frederik Valentin",Buy Corals Online,Buy Corals Online,7.2,0
27,Varg,Varg2TM,Nordic Flora Series Pt. 3: Gore-Tex City,Nordic Flora Series Pt. 3: Gore-Tex City,7.2,0
20,Ryan Adams & the Cardinals,"Ryan Adams, The Cardinals",III/IV,III/IV,6.9,0
28,Martin L. Gore,Martin Gore,Counterfeit 2,Counterfeit 2,6.9,0


### fuzzy text match (same score, close date)

we compare the whole "artist album" string with `SequenceMatcher` (1.0 = identical). every match here gets printed so we can check it by eye.

In [18]:
used_albums |= set(m2["album_id"])
used_reviews |= set(m2["review_tombstone_id"])

left = albums[~albums["album_id"].isin(used_albums)]
pool = p4k_meta[~p4k_meta["review_tombstone_id"].isin(used_reviews)]

# candidates: same score (rounded so 7.3 == 7.3 exactly), then the date window
left = left.assign(score_r=left["score"].round(1))
pool = pool.assign(score_r=pool["score_p4k"].round(1))
cand = left.merge(pool, on="score_r", suffixes=("", "_p4k"))
cand["days_off"] = (cand["reviewdate"] - cand["pub_date"]).dt.days.abs()
cand = cand[cand["days_off"] <= DAYS].copy()

ours = cand["artist_key"] + " " + cand["album_key"]
theirs = cand["artist_key_p4k"] + " " + cand["album_key_p4k"]
cand["similarity"] = [SequenceMatcher(None, a, b).ratio() for a, b in zip(ours, theirs)]

# best-looking candidate first, then the usual one-to-one cleanup
cand = cand[cand["similarity"] >= 0.7].sort_values("similarity", ascending=False)
m3 = cand.drop_duplicates("album_id").drop_duplicates("review_tombstone_id").copy()
m3["match_method"] = "fuzzy"

print("matched:", len(m3))
m3[["artist", "artists", "album", "title", "score", "days_off", "similarity"]].sort_values("similarity")

matched: 1


,artist,artists,album,title,score,days_off,similarity
111,Ólafur Arnalds,Ólafur Arnalds,Now I Am Winter,For Now I Am Winter,5.5,0,0.9375


### put the passes together and attach the review text

In [19]:
keep = ["album_id", "review_tombstone_id", "review_url", "match_method", "days_off"]
matches = pd.concat([m1[keep], m2[keep], m3[keep]], ignore_index=True)

# sanity: nothing matched twice
print("albums matched twice:", matches["album_id"].duplicated().sum())
print("reviews used twice:", matches["review_tombstone_id"].duplicated().sum())

matched = (albums
           .merge(matches, on="album_id", how="left")
           .merge(p4k[["review_tombstone_id", "best_new_music", "word_count", "body"]],
                  on="review_tombstone_id", how="left"))

print()
print(matched["match_method"].value_counts(dropna=False))
print()
print("coverage: {:.1%}".format(matched["body"].notna().mean()))

albums matched twice: 0
reviews used twice: 0

match_method
exact          10723
album+score       39
NaN                1
fuzzy              1
Name: count, dtype: int64

coverage: 100.0%


In [20]:
# what's still unmatched?
unmatched = matched[matched["body"].isna()]
print("unmatched:", len(unmatched))
unmatched[["artist", "album", "reviewdate", "score", "genre"]].head(30)

unmatched: 1


,artist,album,reviewdate,score,genre
3028,Sigur Rós,Agaetis Byrjun,1999-06-01,9.4,Rock


In [21]:
# spot check: 10 random matches, our names next to theirs
check = matched.dropna(subset=["body"]).sample(10, random_state=2)
check = check.merge(p4k_meta[["review_tombstone_id", "artists", "title"]], on="review_tombstone_id")
check[["artist", "artists", "album", "title", "match_method", "review_url"]]

,artist,artists,album,title,match_method,review_url
0,Shipping News,Shipping News,Flies the Fields,Flies the Fields,exact,/reviews/albums/7124-flies-the-fields/
1,Goldfrapp,Goldfrapp,Seventh Tree,Seventh Tree,exact,/reviews/albums/11128-seventh-tree/
2,The View,The View,Hats Off to the Buskers,Hats Off to the Buskers,exact,/reviews/albums/9924-hats-off-to-the-buskers/
3,Shelby Lynne,Shelby Lynne,Just a Little Lovin',Just a Little Lovin',exact,/reviews/albums/11256-just-a-little-lovin/
4,Hiss Golden Messenger,Hiss Golden Messenger,Lateness of Dancers,Lateness of Dancers,exact,/reviews/albums/19681-hiss-golden-messenger-la...
5,Stella Donnelly,Stella Donnelly,Beware of the Dogs,Beware of the Dogs,exact,/reviews/albums/stella-donnelly-beware-of-the-...
6,Ceremony,Ceremony,Zoo,Zoo,exact,/reviews/albums/16374-zoo/
7,jaimie branch,jaimie branch,FLY or DIE II: bird dogs of paradise,FLY or DIE II: bird dogs of paradise,exact,/reviews/albums/jaimie-branch-fly-or-die-ii-bi...
8,John Frusciante,John Frusciante,Inside of Emptiness,Inside of Emptiness,exact,/reviews/albums/3235-inside-of-emptiness/
9,Thao & the Get Down Stay Down,Thao & the Get Down Stay Down,A Man Alive,A Man Alive,exact,/reviews/albums/21415-a-man-alive/


## saving everything

The only unmatched album is Sigur Rós's *Ágætis byrjun*. Its review just isn't in the database (we checked), so we drop it.

In [22]:
reviews = matched.dropna(subset=["body"]).copy()
print("albums with review text:", len(reviews))

albums with review text: 10763



One row per album. Join it back to `pitchfork_discogs_initial.csv` on `artist` + `album` + date (that file's `date` column uses the same YYYY-MM-DD format as our `reviewdate`).

In [27]:
out = reviews.rename(columns={"body": "review_text"})
out["best_new_music"] = out["best_new_music"].astype("boolean")  # 1/0/NaN -> True/False/<NA> (NaN = before 2003, no BNM yet)

save_cols = ["artist", "album", "reviewdate", "score", "genre", "reviewauthor",
             "best_new_music", "review_url", "word_count", "short_review",
             "match_method", "review_text"]
out = out[save_cols]

out.to_csv("reviews_text.csv", index=False)
print(out.shape)
out.drop(columns="review_text").head()

(10763, 12)


,artist,album,reviewdate,score,genre,reviewauthor,best_new_music,review_url,word_count,short_review,match_method
0,Studio 1,Studio 1,2009-02-18,8.5,Electronic,Andy Battaglia,False,/reviews/albums/12682-studio-1/,522.0,False,exact
1,John Fahey,The Great Santa Barbara Oil Slick,2005-02-13,8.2,Folk/Country,Mark Richardson,False,/reviews/albums/2947-the-great-santa-barbara-o...,627.0,False,exact
2,Reigning Sound,Too Much Guitar,2004-08-19,8.3,Electronic,Stephen M. Deusner,False,/reviews/albums/6922-too-much-guitar/,626.0,False,exact
3,Mac Miller,Swimming,2018-08-03,7.5,Rap,Evan Rytlewski,False,/reviews/albums/mac-miller-swimming/,633.0,False,exact
4,The Essex Green,Essex Green EP,2000-05-31,8.1,Rock,Matt LeMay,<NA>,/reviews/albums/2836-essex-green-ep/,376.0,False,exact
